In [ ]:
from algorithm_utils import *
import json
import pandas as pd
from transformers import AutoTokenizer
from collections import Counter
import tree_sitter_java as ts_java
from tree_sitter import Language, Parser

In [ ]:
model = "Qwen/Qwen2.5-Coder-32B"
short_model_name = model.split("/")[-1]

tokenizer = AutoTokenizer.from_pretrained(model)

code_snippets_with_aois_dir = "../code_snippets_with_aois"
code_snippets_with_data_driven_aois_dir = "../code_snippets_with_data_driven_aois"
# Output directories for results
model_results_dir = f"../model_results/{short_model_name}"
output_dir_comparisons = os.path.join(model_results_dir, "data_driven_aois_comparison")

if output_dir_comparisons and not os.path.exists(output_dir_comparisons):
    os.makedirs(output_dir_comparisons)

In [ ]:
results_path = f"{model_results_dir}/perplexity_results_with_aois.json"
if os.path.exists(results_path):
    print(f"Loading existing perplexity results for {short_model_name}.")
    with open(results_path, "r") as f:
        results = json.load(f)
else:
    print("No existing perplexity results found.")

In [ ]:
# read out the optimal aoi region for each file
df = pd.read_csv("../code_snippets_with_aois/aoi_sizes.csv", delimiter=";")
# Filter rows where OPTIMAL is True
optimal_aoi_df = df[df['OPTIMAL']]
# look which aoi region is optimal for each file
optimal_aoi = {}
for index, row in optimal_aoi_df.iterrows():
    file_name = row['Snippet']
    aoi_type = row['NO']
    optimal_aoi[file_name] = aoi_type

# keep only the optimal aoi region for each file in the results
for file_name, file_results in results.items():
    if file_name in optimal_aoi:
        aoi_type = optimal_aoi[file_name]
        results[file_name]["aoi_regions"] = {aoi_type: results[file_name]["aoi_regions"][aoi_type]}
    else:
        results[file_name]["aoi_regions"] = {}

In [ ]:
java_language = Language(ts_java.language())
parser = Parser(language=java_language)

# Initialize counters for each subset
subset_names = ["clean", "obf"]
subset_totals = {name: 0 for name in subset_names}
subset_contains = {name: 0 for name in subset_names}
subset_category_counts = {name: Counter() for name in subset_names}
# Initialize lists to track overlap/non-overlap per subset
overlap_files = {name: [] for name in subset_names}
no_overlap_files = {name: [] for name in subset_names}


def get_subset(file_name):
    if "clean" in file_name.lower():
        return "clean"
    elif "obf" in file_name.lower():
        return "obf"

### Main Algorithm for Confusing Region Detection

In [ ]:
for file_name, snippet in results.items():
    token_results = snippet['results']
    tokens = [tr['token'].strip() for tr in token_results]
    perps = [tr['perplexity'] for tr in token_results]

    # AOI Detection Pipeline
    aois = detect_perplexity_peaks_scipy(tokens, perps, min_span=1, distance=1, prominence=0.8)
    semantic_tokens, index_map = combine_tokens_semantically(tokens)
    aois = expand_aois_semantically(aois, tokens, semantic_tokens, index_map)
    aois = merge_adjacent_aois(aois, tokens)

    # Prepare file paths
    version_folder = file_name.split("-")[2]
    source_file_path = os.path.join("../code_snippets", version_folder, file_name + ".java")

    if not os.path.exists(source_file_path):
        print(f"[!] Missing file: {source_file_path}")
        continue

    with open(source_file_path, "r", encoding="utf-8") as f:
        code = f.read()

    # Token offset mapping
    encoding = tokenizer(code, return_offsets_mapping=True, add_special_tokens=False)
    offsets = encoding['offset_mapping']

    # AST parse
    tree = parser.parse(bytes(code, "utf8"))
    root_node = tree.root_node

    # Filter + Label AOIs
    filtered_aois = []
    aoi_regions = []
    labeled_aois = {}

    # Collect valid AOIs (not Variable / Assignment)
    valid_aois = []
    for aoi in aois:
        start_token, end_token = aoi['start'], aoi['end']
        start_char = offsets[start_token][0]
        end_char = offsets[end_token][1]

        largest = find_largest_fitting_node(root_node, start_char, end_char)
        if not largest:
            # return the node itself if no larger node is found
            largest = root_node
        label = largest.type
        category = categorize_label(label)

        if category in {"Type", "Identifier", "Control Flow", "Punctuation"}:
            continue  # Skip these AOIs

        valid_aois.append({
            'start_token': start_token,
            'end_token': end_token,
            'start_char': start_char,
            'end_char': end_char,
            'peak_value': aoi['peak_value'],
            'category': category,
            'label': label,
            'text': code[start_char:end_char],
        })

    # Find the max peak_value
    if valid_aois:
        max_peak = max(aoi['peak_value'] for aoi in valid_aois)

        threshold = 0  # Keep AOIs with peak_value >= threshold * max_peak
        selected_aois = [aoi for aoi in valid_aois if aoi['peak_value'] >= threshold * max_peak]

        aoi_regions = [[aoi['start_token'], aoi['end_token']] for aoi in selected_aois]
        labeled_aois = {
            str(i + 1): {
                'start_char': aoi['start_char'],
                'end_char': aoi['end_char'],
                'peak_value': aoi['peak_value'],
                'category': aoi['category'],
                'label': aoi['label'],
                'text': aoi['text'],
            }
            for i, aoi in enumerate(selected_aois)
        }

    results[file_name]['aoi_regions_algorithm'] = {'EXP': aoi_regions}
    results[file_name]['aoi_labels'] = labeled_aois

    # Plot result with AOIs
    algo_contains_aoi = plot_token_perplexity(
        results,
        file=file_name,
        output_path=os.path.join(output_dir_comparisons, f"{file_name}_perplexity_plot.pdf"),
    )

    subset = get_subset(file_name)
    subset_totals[subset] += 1
    if algo_contains_aoi:
        subset_contains[subset] += 1
        overlap_files[subset].append(file_name)
    else:
        no_overlap_files[subset].append(file_name)

    # Count AOI categories for this snippet
    if "aoi_labels" in results[file_name]:
        for label_info in results[file_name]["aoi_labels"].values():
            subset_category_counts[subset][label_info["category"]] += 1

    # Insert AOI tags using Hugging Face tokenizer
    code_with_tags = insert_aoi_tags_into_code(code, aoi_regions, tokenizer)

    # Save tagged code
    output_file_path = os.path.join(code_snippets_with_data_driven_aois_dir, version_folder, file_name + ".java")
    os.makedirs(os.path.dirname(output_file_path), exist_ok=True)
    with open(output_file_path, "w", encoding="utf-8") as f:
        f.write(code_with_tags)
    print(f"[✓] Saved AOI-tagged code: {output_file_path}")

    # Save results with updated AOIs and labels
    with open(results_path, "w") as f:
        json.dump(results, f, indent=2)
    print(f"[✓] Saved updated results with AOI labels to {results_path}")

# After processing all files, print summary
overlap_summary = {
    "overlap": overlap_files,
    "no_overlap": no_overlap_files
}

with open(os.path.join(output_dir_comparisons, "aoi_overlap_snippet_lists.json"), "w") as f:
    json.dump(overlap_summary, f, indent=2)
print(
    f"[✓] Saved AOI overlap/non-overlap snippet lists to {os.path.join(output_dir_comparisons, 'aoi_overlap_snippet_lists.json')}")

In [ ]:
# set variables for the number of snippets in each category
statistics = {
    "num_total_clean_manually_labeled": 0,  # Total Clean Snippets
    "num_total_confusing_manually_labeled": 0,  # Total Confusing Snippets
    "num_total_clean_detected": 0,  # Detected Regions in Clean Snippet
    "num_total_confusing_detected": 0,  # Detected Regions in Confusing Snippet
    "num_clean_only": 0,  # Clean Snippets without Detected Regions
    "num_confusing_only": 0,  # Confusing Snippets without Detected Regions
    "num_clean_overlap": 0,  # Clean Snippets with Detected Regions
    "num_confusing_overlap": 0,  # Confusing Snippets with Detected Regions
}

print("\n=== AOI Overlap Summary by Subset ===")
for name in subset_names:
    total = subset_totals[name]
    contains = subset_contains[name]
    percent = (contains / total) * 100 if total > 0 else 0
    if name == "clean":
        statistics["num_total_clean_manually_labeled"] = total
        statistics["num_clean_overlap"] = contains
        statistics["num_clean_only"] = total - contains
    elif name == "obf":
        statistics["num_total_confusing_manually_labeled"] = total
        statistics["num_confusing_overlap"] = contains
        statistics["num_confusing_only"] = total - contains
    print(f"{name.capitalize():8}: {contains}/{total} ({percent:.2f}%)")

overall_total = sum(subset_totals.values())
overall_contains = sum(subset_contains.values())
overall_percent = (overall_contains / overall_total) * 100 if overall_total > 0 else 0
print(f"\nOverall   : {overall_contains}/{overall_total} ({overall_percent:.2f}%)")

# Print AOI category counts per subset
print("\n=== AOI Category Count by Snippet Type ===")
for name in subset_names:
    print(f"\n{name.capitalize()} snippets:")
    total = sum(subset_category_counts[name].values())
    if name == "clean":
        statistics["num_total_clean_detected"] = total
    elif name == "obf":
        statistics["num_total_confusing_detected"] = total
    for category, count in subset_category_counts[name].most_common():
        percent = (count / total) * 100 if total > 0 else 0
        print(f"  {category:25}: {count} ({percent:.1f}%)")
    print(f"  Total AOIs: {total}")  # Gather all AOI labels from all files and categorize them
all_aoi_labels = Counter()
for file_name, snippet in results.items():
    if "aoi_labels" in snippet:
        for label_info in snippet["aoi_labels"].values():
            all_aoi_labels[label_info["category"]] += 1
# Print overall AOI category counts
print("\n=== Overall AOI Category Count by All Snippets ===")
for category, count in all_aoi_labels.most_common():
    percent = (count / sum(all_aoi_labels.values())) * 100 if sum(all_aoi_labels.values()) > 0 else 0
    print(f"{category:25}: {count} ({percent:.1f}%)")

print(f"Total AOIs: {sum(all_aoi_labels.values())}")  # Total AOIs across all snippets

print("\n=== Statistics Summary ===")
for key, value in statistics.items():
    print(f"{key:30}: {value}")

# Save statistics to a JSON file
statistics_path = os.path.join(output_dir_comparisons, "data_driven_aois_overlap_statistics.json")
with open(statistics_path, "w") as f:
    json.dump(statistics, f, indent=2)